# IMPORTS

In [81]:
# import sys
#!{sys.executable} -m pip install torch torchvision torchaudio
#!{sys.executable} -m pip install torch-geometric

In [82]:
import torch
import torch.nn as nn
from dataset_loader import get_dataloaders
from models import build_model

In [ ]:
import random
import numpy as np

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# CONFIG

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATASET_DIR = "datasets"
BATCH_SIZE = 32
EPOCHS = 30
LR = 5e-4

MODEL_NAME = "gat"  # change to "zero" or "gat" or "mlp" or "nnconv"

# LOAD DATA

In [84]:
train_loader, val_loader, test_loader = get_dataloaders(
    DATASET_DIR,
    batch_size=BATCH_SIZE
)

sample_batch = next(iter(train_loader))
print(sample_batch)

Processing...
Done!
Processing...
Done!
Processing...
Done!


DataBatch(x=[442, 13], edge_index=[2, 2960], edge_attr=[2960, 3], y=[442, 4], pos=[442, 3], target=[442, 3], obstacles=[128, 2], formation_id=[32], episode_id=[32], step_idx=[32], num_drones=[32], batch=[442], ptr=[33])


# build model

In [85]:
in_dim = sample_batch.x.shape[1]

model = build_model(MODEL_NAME, in_dim=in_dim).to(DEVICE)

print(model)

GATResidualRegressor(
  (conv1): GATv2Conv(13, 64, heads=4)
  (conv2): GATv2Conv(256, 64, heads=1)
  (mlp_head): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=3, bias=True)
  )
)


# loss + optimizer

In [ ]:
def compute_loss(pred, target):
    return nn.functional.mse_loss(pred, target)

if MODEL_NAME != "zero":
    optimizer = torch.optim.Adam(model.parameters(), lr=LR,weight_decay=1e-5)

# evaluation function

In [ ]:
@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss = 0.0
    total_nodes = 0

    for batch in loader:
        batch = batch.to(DEVICE)
        pred = model(batch)
        loss = compute_loss(pred, batch.target.float())

        num_nodes = batch.x.size(0)
        total_loss += loss.item() * num_nodes
        total_nodes += num_nodes

    return total_loss / total_nodes

In [ ]:
best_val_loss = float("inf")
best_epoch = -1
best_state = None
patience = 5
epochs_without_improvement = 0

train_losses = []
val_losses = []

# training loop

In [ ]:

if MODEL_NAME == "zero":
    print("Zero baseline (no training)")

    train_loss = evaluate(train_loader)
    val_loss = evaluate(val_loader)
    test_loss = evaluate(test_loader)

    print("Train:", train_loss)
    print("Val:", val_loss)
    print("Test:", test_loss)

else:
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0
        total_nodes = 0

        for batch in train_loader:
            batch = batch.to(DEVICE)

            optimizer.zero_grad()

            pred = model(batch)
            loss = compute_loss(pred, batch.target.float())

            loss.backward()
            optimizer.step()

            num_nodes = batch.x.size(0)
            total_loss += loss.item() * num_nodes
            total_nodes += num_nodes

        train_loss = total_loss / total_nodes
        val_loss = evaluate(val_loader)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch + 1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        print(f"Epoch {epoch+1} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

        if epochs_without_improvement >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

    print("\nBest epoch:", best_epoch)
    print("Best val loss:", best_val_loss)

    model.load_state_dict(best_state)
    model = model.to(DEVICE)

    test_loss = evaluate(test_loader)
    print("Test loss with best model:", test_loss)

Epoch 1 | Train: 0.1185 | Val: 0.2334
Epoch 2 | Train: 0.0956 | Val: 0.2515
Epoch 3 | Train: 0.0786 | Val: 0.2649
Epoch 4 | Train: 0.0663 | Val: 0.2460
Epoch 5 | Train: 0.0590 | Val: 0.2558
Epoch 6 | Train: 0.0524 | Val: 0.2581
Epoch 7 | Train: 0.0475 | Val: 0.2515
Epoch 8 | Train: 0.0434 | Val: 0.2495
Epoch 9 | Train: 0.0411 | Val: 0.2445
Epoch 10 | Train: 0.0376 | Val: 0.2540
Epoch 11 | Train: 0.0347 | Val: 0.2592
Epoch 12 | Train: 0.0332 | Val: 0.2544
Epoch 13 | Train: 0.0313 | Val: 0.2661
Epoch 14 | Train: 0.0296 | Val: 0.2627
Epoch 15 | Train: 0.0281 | Val: 0.2602
Epoch 16 | Train: 0.0273 | Val: 0.2703
Epoch 17 | Train: 0.0263 | Val: 0.2562
Epoch 18 | Train: 0.0250 | Val: 0.2541
Epoch 19 | Train: 0.0250 | Val: 0.2489
Epoch 20 | Train: 0.0239 | Val: 0.2566

Best epoch: 1
Best val loss: 0.23340338880162953
Test loss with best model: 0.20022679774603824


In [ ]:
results = {
    "model_name": MODEL_NAME,
    "best_epoch": best_epoch,
    "best_val_loss": best_val_loss,
    "test_loss": test_loss,
    "train_losses": train_losses,
    "val_losses": val_losses,
}

print(results)

if MODEL_NAME != "zero" and best_state is not None:
    torch.save(best_state, f"best_{MODEL_NAME}.pt")
    print(f"Saved best model as best_{MODEL_NAME}.pt")